# Early Prediction of Type 2 Diabetes - BEHRT Fine-Tuning
This notebook trains and evaluates the BEHRT model on the binary classification task (Yes/No). It includes data leakage prevention based on strict temporal censoring.

## Step 1: Set up the Google Colab environment
We grant access to our files in Drive and install the necessary libraries.

In [1]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install the base BEHRT library (older version of HuggingFace)
!pip install pytorch_pretrained_bert


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 11.0 MB/s eta 0:00:00


## Step 2: Imports and Paths

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm  # Optimized progress bar for Colab
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

# --- ABSOLUTE GOOGLE DRIVE PATHS ---
TRAIN_PATH = '/content/drive/MyDrive/Colab Notebooks/BEHRT/data/diabetes_train.parquet'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/BEHRT/data/diabetes_test.parquet'
VOCAB_PATH = '/content/drive/MyDrive/Colab Notebooks/BEHRT/data/vocab.pkl'
MODEL_WEIGHTS_PATH = '/content/drive/MyDrive/Colab Notebooks/BEHRT/saved_models/behrt_pretrain_model.pth'

# --- HYPERPARAMETERS ---
MAX_SEQ_LENGTH = 300
BATCH_SIZE = 32
EPOCHS = 4
LEARNING_RATE = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--- Using processing device: {device} ---")


--- Using processing device: cuda ---


## Step 3: Robust Vocabulary Loading
The model needs to know how to translate a code like 'E11' (Diabetes) into its corresponding numerical ID.

In [3]:
print("Loading vocabulary...")
with open(VOCAB_PATH, 'rb') as f:
    vocab_obj = pickle.load(f)

if isinstance(vocab_obj, dict) and 'token2idx' in vocab_obj:
    word2idx = vocab_obj['token2idx']
    print("Successfully mapped 'token2idx' as the vocabulary.")
else:
    word2idx = vocab_obj
    print("Warning: 'token2idx' key not found, using raw object as vocabulary.")

VOCAB_SIZE = len(word2idx)
print(f"\nSUCCESS! Vocabulary loaded. Total unique codes in the hospital dataset: {VOCAB_SIZE}")

sep_id = word2idx.get('SEP', -1)
if sep_id == -1:
    print("WARNING: The 'SEP' token was not found in the vocabulary.")
else:
    print(f"'SEP' token correctly identified with ID: {sep_id}")

Loading vocabulary...
Successfully mapped 'token2idx' as the vocabulary.

SUCCESS! Vocabulary loaded. Total unique codes in the hospital dataset: 52856
'SEP' token correctly identified with ID: 8289


## Step 4: DataLoader Construction
This transforms the `.parquet` tables into mathematical tensors that the GPU can process.

In [4]:
class BEHRT_Diabetes_Dataset(Dataset):
    def __init__(self, filepath, word2idx, max_len=300):
        print(f"Reading parquet file: {filepath}...")
        self.df = pd.read_parquet(filepath)
        self.word2idx = word2idx
        self.max_len = max_len
        self.sep_id = self.word2idx.get('SEP', 0)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        codes = row['code']
        ages = row['age'] if 'age' in self.df.columns else [0] * len(codes)
        label = row['label']

        # Truncate to maximum length
        if len(codes) > self.max_len:
            codes = codes[-self.max_len:]
            ages = ages[-self.max_len:]

        # Convert codes to IDs
        # --- CINTURÓN DE SEGURIDAD ---
        input_ids = [min(self.word2idx.get(c, 0), VOCAB_SIZE - 1) for c in codes]
        age_ids = [min(int(a) if str(a).isdigit() else 0, 1321) for a in ages]

        # Create Segment IDs (flip between 0 and 1 after each SEP)
        seg_ids = []
        current_seg = 0
        for code_id in input_ids:
            seg_ids.append(current_seg)
            if code_id == self.sep_id:
                current_seg = 1 - current_seg

        attention_mask = [1] * len(input_ids)

        # Padding
        padding_length = self.max_len - len(input_ids)
        if padding_length > 0:
            input_ids = input_ids + [0] * padding_length
            age_ids = age_ids + [0] * padding_length
            seg_ids = seg_ids + [0] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'age_ids': torch.tensor(age_ids, dtype=torch.long),
            'seg_ids': torch.tensor(seg_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.float)
        }

train_dataset = BEHRT_Diabetes_Dataset(TRAIN_PATH, word2idx, MAX_SEQ_LENGTH)
test_dataset = BEHRT_Diabetes_Dataset(TEST_PATH, word2idx, MAX_SEQ_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


Reading parquet file: /content/drive/MyDrive/Colab Notebooks/BEHRT/data/diabetes_train.parquet...
Reading parquet file: /content/drive/MyDrive/Colab Notebooks/BEHRT/data/diabetes_test.parquet...


## Step 5: Load the Pre-trained Model
We connect the weights from Phase 1 (MLM) to a new linear layer designed for binary classification.

In [5]:
from pytorch_pretrained_bert import BertConfig, BertForSequenceClassification, BertModel

class BEHRTForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels):
        super().__init__()
        self.bert = BertModel(config)
        # Ajustamos el tamaño a 1322 para que coincida exactamente con tus pesos pre-entrenados
        self.bert.embeddings.age_embeddings = nn.Embedding(1322, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, input_ids, age_ids, seg_ids, attention_mask):
        inputs_embeds = self.bert.embeddings.word_embeddings(input_ids)
        position_embeds = self.bert.embeddings.position_embeddings(
            torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        )
        token_type_embeds = self.bert.embeddings.token_type_embeddings(seg_ids)

        age_embeds = self.bert.embeddings.age_embeddings(age_ids)

        embeddings = inputs_embeds + position_embeds + token_type_embeds + age_embeds
        embeddings = self.bert.embeddings.LayerNorm(embeddings)
        embeddings = self.bert.embeddings.dropout(embeddings)

        # CAMBIO: Capturamos el resultado sin forzar el desempaquetado de dos variables
        encoder_output = self.bert.encoder(embeddings, attention_mask.unsqueeze(1).unsqueeze(2))

        # Si devuelve una tupla, el primer elemento suele ser la lista de capas (o el tensor final)
        if isinstance(encoder_output, tuple):
            sequence_output = encoder_output[0]
        else:
            sequence_output = encoder_output

        # Si sequence_output es una lista (lista de capas), tomamos la última capa [-1]
        if isinstance(sequence_output, list):
            final_layer = sequence_output[-1]
        else:
            final_layer = sequence_output

        pooled_output = self.bert.pooler(final_layer)

        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)

# Configuración...
config = BertConfig(
    vocab_size_or_config_json_file=VOCAB_SIZE,
    hidden_size=288, num_hidden_layers=6, num_attention_heads=12,
    intermediate_size=512, hidden_act="gelu", hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1, max_position_embeddings=512
)

model = BEHRTForSequenceClassification(config, num_labels=1)

# Cargar pesos
state_dict = torch.load(MODEL_WEIGHTS_PATH, map_location=device)

# Cargamos el modelo
model.load_state_dict(state_dict, strict=False)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()
print("Model ready!")

Model ready!


## Step 6: Training (Fine-Tuning)
We fine-tune the model exclusively on the task of reading early historical context to predict incident diabetes.

In [ ]:
import os
import torch

# Create a folder in your Drive to prevent data loss if it stops
save_dir = '/content/drive/MyDrive/Colab Notebooks/BEHRT/saved_models'
os.makedirs(save_dir, exist_ok=True)

print("========================================================")
print("STARTING FINE-TUNING (DIABETES PREDICTION)")
print("========================================================")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training]")

    for batch in progress_bar:
        b_input_ids = batch['input_ids'].to(device)
        b_age_ids = batch['age_ids'].to(device)
        b_seg_ids = batch['seg_ids'].to(device)
        b_mask = batch['attention_mask'].to(device)
        b_labels = batch['label'].to(device)

        model.zero_grad()

        outputs = model(b_input_ids, age_ids=b_age_ids, seg_ids=b_seg_ids, attention_mask=b_mask)
        logits = outputs[0] if isinstance(outputs, tuple) else outputs
        logits = logits.squeeze(-1) # Flatten tensor

        loss = criterion(logits, b_labels)
        total_loss += loss.item()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Mathematical stability to prevent exploding gradients
        optimizer.step()

        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = total_loss / len(train_loader)
    print(f"---> Epoch {epoch+1} Completed | Average Training Loss: {avg_train_loss:.4f}\n")

    # --- AUTOMATIC SAVING ---
    checkpoint_path = os.path.join(save_dir, f'diabetes_model_epoch_{epoch+1}.pth')
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Checkpoint for epoch {epoch+1} successfully saved at: {checkpoint_path}\n")

STARTING FINE-TUNING (DIABETES PREDICTION)


Epoch 1/4 [Training]:   0%|          | 0/7193 [00:00<?, ?it/s]

---> Epoch 1 Completed | Average Training Loss: 0.0551

Checkpoint for epoch 1 successfully saved at: /content/drive/MyDrive/Colab Notebooks/BEHRT/saved_models/diabetes_model_epoch_1.pth



Epoch 2/4 [Training]:   0%|          | 0/7193 [00:00<?, ?it/s]

---> Epoch 2 Completed | Average Training Loss: 0.0372

Checkpoint for epoch 2 successfully saved at: /content/drive/MyDrive/Colab Notebooks/BEHRT/saved_models/diabetes_model_epoch_2.pth



Epoch 3/4 [Training]:   0%|          | 0/7193 [00:00<?, ?it/s]

---> Epoch 3 Completed | Average Training Loss: 0.0322

Checkpoint for epoch 3 successfully saved at: /content/drive/MyDrive/Colab Notebooks/BEHRT/saved_models/diabetes_model_epoch_3.pth



Epoch 4/4 [Training]:   0%|          | 0/7193 [00:00<?, ?it/s]

---> Epoch 4 Completed | Average Training Loss: 0.0269

Checkpoint for epoch 4 successfully saved at: /content/drive/MyDrive/Colab Notebooks/BEHRT/saved_models/diabetes_model_epoch_4.pth



## Step 7: The Final Exam (Test Set Evaluation)
We evaluate the model on patients it has never seen and calculate the scientific metrics for the thesis report.

In [ ]:
import os
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from tqdm import tqdm

# 1. Create a folder for final results in your Google Drive
results_dir = '/content/drive/MyDrive/Colab Notebooks/BEHRT/results'
os.makedirs(results_dir, exist_ok=True)

print("========================================================")
print("EVALUATING MODEL ON TEST SET (DATA LEAKAGE PREVENTED)")
print("========================================================")

model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        b_input_ids = batch['input_ids'].to(device)
        b_age_ids = batch['age_ids'].to(device)
        b_seg_ids = batch['seg_ids'].to(device)
        b_mask = batch['attention_mask'].to(device)
        b_labels = batch['label'].cpu().numpy()

        outputs = model(b_input_ids, age_ids=b_age_ids, seg_ids=b_seg_ids, attention_mask=b_mask)
        logits = outputs[0] if isinstance(outputs, tuple) else outputs
        logits = logits.squeeze(-1)

        # We use Sigmoid to convert logits into probabilities between 0 and 1
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)

        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(b_labels)

# --- PANIC-PROOF SAVING ---
np.save(os.path.join(results_dir, 'all_labels.npy'), all_labels)
np.save(os.path.join(results_dir, 'all_preds.npy'), all_preds)

# --- CALCULATING METRICS ---
auc_roc = roc_auc_score(all_labels, all_probs)
f1 = f1_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)

# Calculate Specificity from Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
TN, FP, FN, TP = cm.ravel()
specificity = TN / (TN + FP)

print("\n========================================================")
print("FINAL SCIENTIFIC RESULTS (INCIDENT DISEASE PREDICTION)")
print("========================================================")
print(f"ROC-AUC Score : {auc_roc:.4f} (Ability to rank patient risk correctly)")
print(f"F1-Score      : {f1:.4f} (Harmonic mean of Precision and Recall)")
print(f"Precision     : {precision:.4f} (Out of all predicted diabetics, how many were true diabetics?)")
print(f"Recall        : {recall:.4f} (Out of all true diabetics, how many did the model catch?)")
print(f"Specificity   : {specificity:.4f} (Out of all true healthy patients, how many were predicted healthy?)")
print("========================================================")
print("Congratulations! You have completed the predictive pipeline for your thesis.")

## Step 8: The Confusion Matrix


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

results_dir = '/content/drive/MyDrive/Colab Notebooks/BEHRT/results'

# Reuse the confusion matrix 'cm' calculated in the previous cell
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Healthy (0)', 'Predicted Diabetic (1)'],
            yticklabels=['True Healthy (0)', 'True Diabetic (1)'],
            annot_kws={"size": 14})

plt.title('Confusion Matrix - Type 2 Diabetes Prediction', fontsize=16, pad=15)
plt.xlabel('Model Prediction', fontsize=14, labelpad=10)
plt.ylabel('Actual Ground Truth', fontsize=14, labelpad=10)
plt.tight_layout()

# Save image directly to Google Drive
cm_path = os.path.join(results_dir, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=300)
plt.show()

print(f" Confusion matrix successfully generated and saved at: {cm_path}")